# AI Assignment by Subhayan Das

## Import libraries

In [1]:
import pygame
import random
import time
import sys
from collections import deque
import numpy as np
import pandas as pd
import os
import sys
from heapq import heappop, heappush
import copy
from PIL import Image, ImageDraw

pygame 2.6.1 (SDL 2.28.4, Python 3.9.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Constants

In [2]:
# Constants
GRID_SIZE = 30
CELL_SIZE = 5
WINDOW_SIZE = CELL_SIZE * (2 * GRID_SIZE + 1)
BACKGROUND_COLOR = (255, 255, 255)
WALL_COLOR = (0, 0, 0)
PATH_COLOR = (255, 255, 255)
START_COLOR = (0, 255, 0)
END_COLOR = (255, 0, 0)
PATHFIND_COLOR = (0, 0, 255)
VISITED_COLOR = (255, 255, 0)


## Path Finding Functions

In [3]:
def calculate_metrics(maze, start, end, screen, traversal_order, explored_nodes, path, start_time):
    def path_complexity(maze):
        dead_ends = 0
        branching_points = 0
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        
        for r in range(1, len(maze) - 1, 2):
            for c in range(1, len(maze[0]) - 1, 2):
                if maze[r][c] == 0:
                    open_neighbors = 0
                    for dr, dc in directions:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
                            open_neighbors += 1
                    if open_neighbors == 0:
                        dead_ends += 1
                    elif open_neighbors > 2:
                        branching_points += 1
        return dead_ends, branching_points

    dead_ends, branching_points = path_complexity(maze)

    total_cells = len(maze) * len(maze[0])
    walls = sum(row.count(1) for row in maze)
    open_cells = total_cells - walls
    wall_density = walls / total_cells

    execution_time = time.time() - start_time

    final_path_length = len(path)

    memory_usage = sys.getsizeof(maze) + sys.getsizeof(path) + sys.getsizeof(explored_nodes)

    return {
        "Algorithm" : "A star",
        "Grid Size" : GRID_SIZE,
        "Path Complexity (Entropy)": (dead_ends, branching_points),
        "Wall Density (Open Space Ratio)": wall_density,
        "Number of Explored Nodes (Search Cost)": len(explored_nodes),
        "Final Path Length": final_path_length,
        "Execution Time (Seconds)": execution_time,
        "Memory Usage (Bytes)": memory_usage
    }

In [4]:
# Manhattan Distance
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star_traversal(maze, screen):
    start_time = time.time()
    start = (1, 1)
    end = (len(maze) - 2, len(maze[0]) - 2)
    
    open_set = [(0, start)]
    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic(start, end)}
    explored_nodes = set()
    traversal_order = []
    
    while open_set:
        _, current = heappop(open_set)
        traversal_order.append(current)
        explored_nodes.add(current)

        if current == end:
            break
        
        for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            neighbor = (current[0] + dr, current[1] + dc)
            if maze[neighbor[0]][neighbor[1]] == 0:
                temp_g_score = g_score[current] + 1
                if neighbor not in g_score or temp_g_score < g_score[neighbor]:
                    g_score[neighbor] = temp_g_score
                    f_score[neighbor] = temp_g_score + heuristic(neighbor, end)
                    heappush(open_set, (f_score[neighbor], neighbor))
                    came_from[neighbor] = current

        draw_maze(screen, maze, [], visited_cells=traversal_order)
        pygame.display.flip()
        pygame.time.delay(1)
    
    path = []
    current = end
    while current in came_from:
        path.append(current)
        current = came_from[current]
    path.reverse()

    # Draw the final path in blue
    draw_maze(screen, maze, path)
    pygame.display.flip()

    return path, traversal_order, explored_nodes

In [5]:
def draw_maze(screen, maze, path, visited_cells=[]):
    for r in range(len(maze)):
        for c in range(len(maze[0])):
            color = WALL_COLOR if maze[r][c] == 1 else PATH_COLOR
            pygame.draw.rect(screen, color, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in visited_cells:
        pygame.draw.rect(screen, VISITED_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in path:
        pygame.draw.rect(screen, PATHFIND_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    pygame.draw.rect(screen, START_COLOR, (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE))
    pygame.draw.rect(screen, END_COLOR, ((len(maze[0]) - 2) * CELL_SIZE, (len(maze) - 2) * CELL_SIZE, CELL_SIZE, CELL_SIZE))


In [6]:
def export_path(maze, path, start, end, filename):
    rows, cols = len(maze), len(maze[0])
    image = Image.new("RGB", (cols * CELL_SIZE, rows * CELL_SIZE), (255, 255, 255))
    draw = ImageDraw.Draw(image)

    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == 1:  # Wall
                draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 0))

    for r, c in path:
        draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 255))

    start_r, start_c = start
    draw.rectangle([start_c * CELL_SIZE, start_r * CELL_SIZE, (start_c + 1) * CELL_SIZE, (start_r + 1) * CELL_SIZE], fill=(0, 255, 0))

    end_r, end_c = end
    draw.rectangle([end_c * CELL_SIZE, end_r * CELL_SIZE, (end_c + 1) * CELL_SIZE, (end_r + 1) * CELL_SIZE], fill=(255, 0, 0))

    image.save(filename)
    print(f"Final path image saved as {filename}")

## Main Function

In [7]:
def main():
    pygame.init()

    screen_astar = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))
    pygame.display.set_caption("Astar Maze Traversal")

    maze = np.load(f"./Mazes/{GRID_SIZE}.npy").tolist()

    start_time_astar = time.time()
    astar_path, astar_traversal_order, astar_visited = a_star_traversal(maze, screen_astar)
    astar_metrics = calculate_metrics(maze, (1, 1), (len(maze) - 2, len(maze[0]) - 2), screen_astar, astar_traversal_order, astar_visited, astar_path, start_time_astar)

    
    print("A star Metrics:")
    for metric, value in astar_metrics.items():
        print(f"{metric}: {value}")

    # Saving the metrics:
    filename = 'results.csv'
    df = pd.DataFrame([astar_metrics])

    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        df = pd.concat([existing_df, df], ignore_index=True)
    
    df.to_csv(filename, index=False)
    
    export_path(maze, astar_path, (1, 1), (len(maze) - 2, len(maze[0]) - 2), f"./Paths/Astar_{GRID_SIZE}.png")
    

    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        pygame.display.flip()

    pygame.quit()

In [8]:

if __name__ == "__main__":
    main()

A star Metrics:
Algorithm: A star
Grid Size: 30
Path Complexity (Entropy): (0, 485)
Wall Density (Open Space Ratio): 0.44396667562483205
Number of Explored Nodes (Search Cost): 1266
Final Path Length: 120
Execution Time (Seconds): 3.0016446113586426
Memory Usage (Bytes): 132912
Final path image saved as ./Paths/Astar_30.png
